# EuroSAT Deep Learning Project

## Stage 1 — Dataset Preparation and Baseline CNN
Satellite Land-Use Classification using CNN Optimization and Transfer Learning.

## 1.1 — Environment Setup

Install the libraries required for data processing, model training, and evaluation.

In [10]:
%pip install torch torchvision matplotlib scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


### Library Verification

Verify that the required packages are installed and accessible.

In [11]:
import torch
import torchvision
import matplotlib
import sklearn
import pandas

print("Everything is installed correctly.")
print("PyTorch:", torch.__version__)

Everything is installed correctly.
PyTorch: 2.7.0+cpu


## 1.2 — Imports and Reproducibility

Import the required modules, set a fixed random seed, and configure the training device.

In [12]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

import numpy as np
import random
from pathlib import Path

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)

PyTorch version: 2.7.0+cpu
Device: cpu


## 1.3 — EuroSAT Dataset

Load the EuroSAT dataset and inspect its size and class labels.

In [13]:
DATA_DIR = Path("../data")

transform = transforms.Compose([
    transforms.ToTensor()
])

dataset = datasets.EuroSAT(
    root=DATA_DIR,
    transform=transform,
    download=True
)

print("Dataset size:", len(dataset))
print("Classes:", dataset.classes)
print("Number of classes:", len(dataset.classes))

Dataset size: 27000
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Number of classes: 10


## 1.4 — Train, Validation, and Test Split

Split the dataset into 70% training, 15% validation, and 15% test sets using a fixed seed.

In [14]:
from torch.utils.data import random_split, DataLoader

train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

generator = torch.Generator().manual_seed(SEED)

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=generator
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 18900
Validation: 4050
Test: 4050


DataLoaders

Create DataLoaders for automated batching and data iteration.Since currently on CPU, use num_workers=0 

In [15]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 296
Validation batches: 64
Test batches: 64


### Data Pipeline Verification

Inspect one training batch to verify the image and label dimensions.

In [16]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Image range:", images.min().item(), "to", images.max().item())

Images shape: torch.Size([64, 3, 64, 64])
Labels shape: torch.Size([64])
Image range: 0.06666667014360428 to 1.0


## 1.5 — Baseline CNN Architecture

Define a simple three-block convolutional neural network from scratch for 10-class classification.

The baseline CNN cell:

In [18]:
import torch.nn as nn

class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = BaselineCNN().to(device)

print(model)

BaselineCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=8192, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=10, bias=True)
  )
)


### Forward-Pass Verification

Run one batch through the network to confirm that the model produces 10 class scores per image.

In [19]:
outputs = model(images.to(device))

print("Output shape:", outputs.shape)

Output shape: torch.Size([64, 10])


## 1.6 — Baseline Model Training

Train and validate the baseline CNN while recording loss and accuracy and saving the best model checkpoint.

### Training Configuration

Define the loss function and optimizer used to train the baseline CNN.

In [20]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)

Loss function: CrossEntropyLoss()
Optimizer: Adam


### Training and Validation Function

Define the training loop, validation procedure, metric tracking, and checkpoint saving.

In [21]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs):

    train_losses = []
    val_losses = []
    val_accuracies = []

    best_val_accuracy = 0.0

    for epoch in range(epochs):

        # ----- TRAINING -----
        model.train()

        running_train_loss = 0.0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * images.size(0)

        train_loss = running_train_loss / len(train_loader.dataset)


        # ----- VALIDATION -----
        model.eval()

        running_val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                loss = criterion(outputs, labels)

                running_val_loss += loss.item() * images.size(0)

                _, predicted = torch.max(outputs, 1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()


        val_loss = running_val_loss / len(val_loader.dataset)
        val_accuracy = 100 * correct / total


        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)


        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {train_loss:.4f} "
            f"Val Loss: {val_loss:.4f} "
            f"Val Accuracy: {val_accuracy:.2f}%"
        )


        # Save best baseline
        if val_accuracy > best_val_accuracy:

            best_val_accuracy = val_accuracy

            torch.save(
                model.state_dict(),
                "../checkpoints/baseline.pth"
            )

    print("\nTraining finished.")
    print(f"Best Validation Accuracy: {best_val_accuracy:.2f}%")

    return train_losses, val_losses, val_accuracies

### Baseline Training Run

Run a short two-epoch baseline training to verify that the complete pipeline works correctly.

In [22]:
EPOCHS = 2

train_losses, val_losses, val_accuracies = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    EPOCHS
)

Epoch [1/2] Train Loss: 1.1582 Val Loss: 0.8931 Val Accuracy: 68.05%
Epoch [2/2] Train Loss: 0.7268 Val Loss: 0.7235 Val Accuracy: 74.15%

Training finished.
Best Validation Accuracy: 74.15%


### Initial Baseline Result

The two-epoch baseline reached a best validation accuracy of **74.15%**. This run confirms that the training pipeline is functioning correctly before longer experiments are performed.